# Titanic Survival Prediction

## Project Overview

This notebook develops a supervised machine-learning model to predict Titanic passenger survival.

**Primary objective:** prioritize recall while comparing multiple classification models and selecting a model with strong generalization.

### Project Workflow
1. Data Loading & Understanding
2. Target and Feature Preparation
3. Train/Test Split
4. Preprocessing Pipeline
5. Logistic Regression Baseline
6. Model Comparison
7. Cross-Validation
8. Hyperparameter Tuning
9. Final Model Evaluation
10. Feature Importance
11. Business Insights


## 1. Data Loading & Initial Inspection

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")
df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())

## 2. Target and Feature Preparation

In [ ]:
X = df.drop('Survived' , axis = 1)
y = df['Survived']


## 3. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(X , y , random_state = 42 , test_size = 0.2 , stratify = y)

## 4. Preprocessing and Logistic Regression

The preprocessing workflow handles numerical and categorical variables through `Pipeline` and `ColumnTransformer`.

- Numerical features: median imputation + standardization
- Categorical features: most-frequent imputation + one-hot encoding
- Logistic Regression is used as the baseline classifier.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score , confusion_matrix , roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

numerical_features = ['Age' , 'Fare']
categorical_features = ['Sex' , 'Embarked' , 'Pclass' ]

numerical_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy = 'median')) , 
    ('scaler' , StandardScaler()) 
])

categorical_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy = 'most_frequent')) , 
    ('encoder' , OneHotEncoder(handle_unknown = 'ignore' , sparse_output = False))
])

preprocessor = ColumnTransformer([
    ('num' , numerical_pipeline , numerical_features) , 
    ('cat' , categorical_pipeline , categorical_features)
])

pipeline = Pipeline([
    ('preprocessor' , preprocessor) , 
    ('model', LogisticRegression() )
])

pipeline.fit(X_train , y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:,1]
accuracy = accuracy_score(y_test , y_pred)
precision = precision_score(y_test , y_pred)
recall = recall_score(y_test , y_pred)
f1 = f1_score(y_test , y_pred)
cm = confusion_matrix(y_test , y_pred)
roc_auc = roc_auc_score(y_test ,y_proba)
print(f'Accuracy : {accuracy} \n Precision : {precision} \n Recall : {recall} \n F1 Score : {f1} \n Confusion Matrix : {cm} \n ROC-AUC : {roc_auc}')

scores = cross_val_score(pipeline , X , y , cv = 5 , scoring = 'recall')
print(f'Scores for each fold : {scores}')
print(f'Mean Recall : {np.mean(scores):.4f}')
print(f"Standard Deviation : {np.std(scores):.4f}")

param_grid = {'model__C' : [0.01 , 0.1 , 1, 10 , 100]}
grid_search = GridSearchCV(pipeline , param_grid , cv = 5 , scoring = 'recall') 
grid_search.fit(X_train , y_train )

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred_tuned = grid_search.predict(X_test)
y_proba_tuned = grid_search.predict_proba(X_test)[:,1]
tuned_recall = recall_score(y_test , y_pred_tuned)
print('Tuned Recall Score is : ', tuned_recall)

## 5. Baseline Model Comparison

The following models are compared using the same preprocessing workflow:

- Logistic Regression
- Decision Tree
- Random Forest
- KNN
- Gaussian Naive Bayes

Evaluation metrics:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score , confusion_matrix , roc_auc_score 
import pandas as pd

decision_pipeline = Pipeline([
    ('preprocessor' , preprocessor) , 
    ('model' , DecisionTreeClassifier(random_state = 42))
])

random_pipeline = Pipeline([
    ('preprocessor'  , preprocessor) , 
    ('model' , RandomForestClassifier(random_state = 42))
])

knn_pipeline = Pipeline([
    ('preprocessor' , preprocessor) , 
    ('model' , KNeighborsClassifier())
])

nb_pipeline = Pipeline([
    ('preprocessor' , preprocessor) , 
    ('model' , GaussianNB())
])

decision_pipeline.fit(X_train , y_train)
random_pipeline.fit(X_train , y_train)
knn_pipeline.fit(X_train , y_train)
nb_pipeline.fit(X_train , y_train)

y_pred_dt = decision_pipeline.predict(X_test)
y_pred_rf = random_pipeline.predict(X_test)
y_pred_knn = knn_pipeline.predict(X_test)
y_pred_nb = nb_pipeline.predict(X_test)
y_proba_dt = decision_pipeline.predict_proba(X_test)[:, 1]
y_proba_rf = random_pipeline.predict_proba(X_test)[:, 1]
y_proba_knn = knn_pipeline.predict_proba(X_test)[:, 1]
y_proba_nb = nb_pipeline.predict_proba(X_test)[:, 1]

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test , y_pred_dt)
recall_dt = recall_score(y_test , y_pred_dt)
f1_dt = f1_score(y_test , y_pred_dt)
cm_dt = confusion_matrix(y_test , y_pred_dt)
roc_auc_dt = roc_auc_score(y_test ,y_proba_dt)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test , y_pred_rf)
recall_rf = recall_score(y_test , y_pred_rf)
f1_rf = f1_score(y_test , y_pred_rf)
cm_rf = confusion_matrix(y_test , y_pred_rf)
roc_auc_rf = roc_auc_score(y_test ,y_proba_rf)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test , y_pred_knn)
recall_knn = recall_score(y_test , y_pred_knn)
f1_knn = f1_score(y_test , y_pred_knn)
cm_knn = confusion_matrix(y_test , y_pred_knn)
roc_auc_knn = roc_auc_score(y_test ,y_proba_knn)

accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test , y_pred_nb)
recall_nb = recall_score(y_test , y_pred_nb)
f1_nb = f1_score(y_test , y_pred_nb)
cm_nb = confusion_matrix(y_test , y_pred_nb)
roc_auc_nb = roc_auc_score(y_test ,y_proba_nb)

df_metric = pd.DataFrame({"Model" : ['Logistic Regression' , 'Decision Tree' , 'Random Forest' , 'KNN' , 'GaussianNB'] , 
                        "Accuracy Score" : [accuracy , accuracy_dt ,accuracy_rf ,accuracy_knn ,accuracy_nb ] , 
                         'Precision Score' : [precision , precision_dt , precision_rf , precision_knn , precision_nb ] , 
                         'Recall Score' : [recall , recall_dt , recall_rf , recall_knn , recall_nb ] , 
                         "F1 Score" : [f1 , f1_dt , f1_rf , f1_knn , f1_nb ] , 
                         'ROC-AUC Score' : [roc_auc , roc_auc_dt , roc_auc_rf , roc_auc_knn , roc_auc_nb ]})
df_metric

## 6. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

print("\nLogistic Regression :")
scores_lr = cross_val_score(pipeline , X , y ,cv = 5 ,scoring = 'recall')
print('Scores :' , scores_lr)
print('Scores Mean : ' , np.mean(scores_lr))
print(f"Standard Deviation : {np.std(scores_lr)}\n")

print("Decision Tree:")
scores_dt = cross_val_score(decision_pipeline , X , y ,cv = 5 ,scoring = 'recall')
print('Scores :' , scores_dt)
print('Scores Mean : ' , np.mean(scores_dt))
print(f"Standard Deviation : {np.std(scores_dt)}\n")

print("Random Forest:")
scores_rf = cross_val_score(random_pipeline , X , y  , cv = 5 , scoring = 'recall')
print('Scores :' , scores_rf)
print('Scores Mean : ' , np.mean(scores_rf))
print(f'Standard Deviation : {np.std(scores_rf)}\n')

print("KNN :")
scores_knn = cross_val_score(knn_pipeline , X , y , cv = 5 , scoring = 'recall')
print('Scores :' , scores_knn)
print('Scores Mean : ' , np.mean(scores_knn))
print(f'Standard Deviation : {np.std(scores_knn)}\n')

print("Gaussian NB:")
scores_nb = cross_val_score(nb_pipeline , X , y , cv = 5 , scoring = 'recall')
print('Scores :' , scores_nb)
print('Scores Mean : ' , np.mean(scores_nb))
print(f'Standard Deviation : {np.std(scores_nb)}\n')


## 7. Hyperparameter Tuning — Random Forest

The Random Forest is tuned with `GridSearchCV` using **recall** as the scoring metric because recall is the primary project objective.


In [ ]:
param_grid_rf = {
    'model__n_estimators' : [100 , 200 , 300] , 
    'model__max_depth' : [3,5,7,10,None] , 
    'model__min_samples_leaf' : [1,2,4] , 
    'model__min_samples_split' : [2,5,10]
}

grid_rf = GridSearchCV(
    random_pipeline , 
    param_grid_rf , 
    cv = 5 , 
    scoring = 'recall' , 
    n_jobs  = -1
)

grid_rf.fit(X_train , y_train)

print("Best Parameters : " , grid_rf.best_params_)
print('Best CV Recall : ' , grid_rf.best_score_)

## 8. Tuned Random Forest — Test Evaluation

In [ ]:
y_pred_rf_tuned = grid_rf.predict(X_test)

y_proba_rf_tuned = grid_rf.predict_proba(X_test)[:, 1]

accuracy_rf_tuned = accuracy_score(y_test, y_pred_rf_tuned)
precision_rf_tuned = precision_score(y_test, y_pred_rf_tuned)
recall_rf_tuned = recall_score(y_test, y_pred_rf_tuned)
f1_rf_tuned = f1_score(y_test, y_pred_rf_tuned)
roc_auc_rf_tuned = roc_auc_score(y_test, y_proba_rf_tuned)

print(f"Accuracy: {accuracy_rf_tuned:.4f}")
print(f"Precision: {precision_rf_tuned:.4f}")
print(f"Recall: {recall_rf_tuned:.4f}")
print(f"F1 Score: {f1_rf_tuned:.4f}")
print(f"ROC-AUC: {roc_auc_rf_tuned:.4f}")

## 9. Hyperparameter Tuning — Gaussian Naive Bayes

`var_smoothing` is tuned using `GridSearchCV`, again using recall as the scoring metric.


In [ ]:
param_grid_nb = {
    'model__var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

grid_nb = GridSearchCV(
    nb_pipeline,
    param_grid_nb,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_nb.fit(X_train, y_train)

print("Best Parameters:", grid_nb.best_params_)
print("Best CV Recall:", grid_nb.best_score_)

## 10. Tuned Gaussian Naive Bayes — Test Evaluation

In [ ]:
y_pred_nb_tuned = grid_nb.predict(X_test)
y_proba_nb_tuned = grid_nb.predict_proba(X_test)[:, 1]

accuracy_nb_tuned = accuracy_score(y_test, y_pred_nb_tuned)
precision_nb_tuned = precision_score(y_test, y_pred_nb_tuned)
recall_nb_tuned = recall_score(y_test, y_pred_nb_tuned)
f1_nb_tuned = f1_score(y_test, y_pred_nb_tuned)
roc_auc_nb_tuned = roc_auc_score(y_test, y_proba_nb_tuned)

print(f"Accuracy: {accuracy_nb_tuned:.4f}")
print(f"Precision: {precision_nb_tuned:.4f}")
print(f"Recall: {recall_nb_tuned:.4f}")
print(f"F1 Score: {f1_nb_tuned:.4f}")
print(f"ROC-AUC: {roc_auc_nb_tuned:.4f}")

## 11. Feature Importance — Final Random Forest

Feature importance is extracted from the Random Forest after preprocessing so that one-hot encoded feature names can be mapped to their importance values.


In [ ]:
feature_names = random_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = random_pipeline.named_steps["model"].feature_importances_
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print(feature_importance_df)
feature_importance_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance_df.head(10).sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 10 Random Forest Feature Importances")
plt.tight_layout()
plt.show()

## 12. Business / Data Insights

### 1. Gender and Survival
Female passengers had a substantially higher survival rate than male passengers. Gender was therefore an important variable for predicting survival.

### 2. Passenger Class and Survival
Passenger class was strongly associated with survival. First-class passengers generally had better survival outcomes than passengers in lower classes.

### 3. Age and Survival
Age was an important predictor of survival. The Random Forest also identified `Age` as one of its most important features.

### 4. Fare and Survival
Fare was the most important individual feature in the Random Forest model with an importance of approximately 0.296.

### 5. Family Structure
Family-related features such as `FamilySize` and `IsAlone` can provide additional information about passenger survival patterns.


## 13. Final Model Summary

Based on the results produced in this notebook, the **baseline Random Forest** is the selected final model.

The selection prioritizes recall and considers overall test performance and generalization rather than choosing a model solely by accuracy.

### Final Random Forest Results
- Accuracy: **0.8156**
- Precision: **0.7813**
- Recall: **0.7246**
- F1 Score: **0.7519**
- ROC-AUC: **0.8442**
- Mean 5-Fold CV Recall: **0.7454**

> The tuned Random Forest was not selected because its test recall decreased relative to the baseline model.
